# Supplementary Figure 8 | Allele frequency, effect size, and lead-variant distance of cis-QTLs

**a.** Densities of minor allele frequency (MAF) for SuSiE eQTL credible-set lead variants (highest-PIP variant per c.s.), pooled across tissues and stratified by biotype (novel lncRNA, existing lncRNA, protein-coding, and pseudogene).

**b.** Densities of absolute normalized effect size (|β|) for SuSiE eQTL c.s. lead variants.

**c.** Densities of absolute genomic distance from the SuSiE eQTL c.s. lead variant to the nearest transcript TSS of the same gene.

**d.** Densities of MAF for SuSiE sQTL c.s. lead variants.

**e.** Densities of absolute normalized effect size (|β|) for SuSiE sQTL c.s. lead variants.

**f.** Densities of absolute genomic distance from the SuSiE sQTL c.s. lead variant to the nearer end of the LeafCutter intron phenotype.


## Setup


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 4.8)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

fig_width, fig_height, fig_dpi = 7.2, 5.2, 300
kde_min_n = 10

biotype_order = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene"]
biotype_pretty = {
    "novel_lncRNA": "novel lncRNA",
    "existing_lncRNA": "existing lncRNA",
    "protein_coding": "protein-coding",
    "pseudogene": "pseudogene",
}
paired = sns.color_palette("Paired", 12)
biotype_palette = {
    "novel_lncRNA": paired[5],
    "existing_lncRNA": paired[4],
    "protein_coding": paired[1],
    "pseudogene": paired[3],
}


## Paths


In [ ]:
# Open inputs (portal SuSiE / signif pairs + GENCODE)
gencode_class = Path("path/to/gencode_v39_v47_gene_classification.txt")
qtl_tissue_ids = Path("path/to/qtl_tissue_ids.txt")
eqtl_dir = Path("path/to/v11_eqtl_susie")
sqtl_dir = Path("path/to/v11_sqtl_susie")
gencode_gtf = Path("path/to/gencode.v47.annotation.gtf")

fig_dir = Path("figures")


## Load data


### QTL tissues, biotype map, and transcript TSS (from GTF)


In [ ]:
tissues = pd.read_table(qtl_tissue_ids, header=None)[0].tolist()
print(f"Tissues: {len(tissues)}")

gencode = pd.read_table(gencode_class, low_memory=False)
gencode = gencode[gencode["gene_type_grouped"].isin(biotype_order)].copy()
id_to_biotype = (
    gencode.dropna(subset=["gene_id_v47"])
    .drop_duplicates("gene_id_v47")
    .set_index("gene_id_v47")["gene_type_grouped"]
)
print(id_to_biotype.value_counts().reindex(biotype_order).to_string())

# Strand-aware transcript TSSs from GENCODE v47 annotation GTF (1-based inclusive).
attr_re = re.compile(r'(\w+)\s+"([^"]+)"')
tss_lists = {}
n_tx = 0
with open(gencode_gtf) as fh:
    for line in tqdm(fh, desc=f"transcript TSS {gencode_gtf.name}"):
        if not line or line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) < 9 or fields[2] != "transcript":
            continue
        start1, end1, strand = int(fields[3]), int(fields[4]), fields[6]
        attrs = dict(attr_re.findall(fields[8]))
        gene_id = attrs.get("gene_id", "")
        if not gene_id:
            continue
        tss = start1 if strand == "+" else end1
        tss_lists.setdefault(gene_id, []).append(tss)
        n_tx += 1
gene_tx_tss = {gid: np.asarray(v, dtype=np.int64) for gid, v in tss_lists.items()}
print(f"Genes with transcript TSS: {len(gene_tx_tss):,}; transcripts: {n_tx:,}")


### Build SuSiE eQTL credible-set leads


In [ ]:
def min_abs_tx_tss_dist(gene_id, variant_pos):
    tss = gene_tx_tss.get(gene_id)
    if tss is None or len(tss) == 0:
        return np.nan
    return float(np.min(np.abs(tss - int(variant_pos))))


parts = []
for tissue in tqdm(tissues, desc="eQTL SuSiE leads"):
    susie_path = eqtl_dir / f"{tissue}.v11.eQTLs.SuSiE_summary.parquet"
    pairs_path = eqtl_dir / f"{tissue}.v11.eQTLs.signif_pairs.parquet"
    if not susie_path.is_file():
        raise FileNotFoundError(f"Missing eQTL SuSiE summary: {susie_path}")
    if not pairs_path.is_file():
        raise FileNotFoundError(f"Missing eQTL signif pairs: {pairs_path}")

    susie = pd.read_parquet(
        susie_path,
        columns=["phenotype_id", "variant_id", "pip", "af", "afc", "cs_id", "cs_size"],
    )
    lead_idx = susie.groupby(["phenotype_id", "cs_id"], sort=False)["pip"].idxmax()
    leads = susie.loc[lead_idx].copy()
    leads["gene_id"] = leads["phenotype_id"]
    leads["biotype"] = leads["gene_id"].map(id_to_biotype)
    leads = leads[leads["biotype"].isin(biotype_order)].copy()
    leads["variant_pos"] = leads["variant_id"].str.split("_").str[1].astype(int)

    pairs = pd.read_parquet(pairs_path, columns=["phenotype_id", "variant_id", "slope"])
    df = leads.merge(pairs, on=["phenotype_id", "variant_id"], how="left")
    df["min_abs_tx_tss_dist"] = [
        min_abs_tx_tss_dist(gid, pos) for gid, pos in zip(df["gene_id"], df["variant_pos"])
    ]
    df["tissue"] = tissue
    parts.append(
        df[
            [
                "tissue",
                "phenotype_id",
                "gene_id",
                "variant_id",
                "cs_id",
                "cs_size",
                "pip",
                "af",
                "afc",
                "slope",
                "variant_pos",
                "min_abs_tx_tss_dist",
                "biotype",
            ]
        ]
    )

eqtl_df = pd.concat(parts, ignore_index=True)


### SuSiE eQTL credible-set leads


In [ ]:
print(f"eQTL CS-lead×tissue rows: {len(eqtl_df):,}")
print(eqtl_df["biotype"].value_counts().reindex(biotype_order).to_string())
print(
    f"missing slope: {eqtl_df['slope'].isna().sum():,} ({100 * eqtl_df['slope'].isna().mean():.1f}%)"
)
print(f"missing transcript-TSS distance: {eqtl_df['min_abs_tx_tss_dist'].isna().sum():,}")


### Build SuSiE sQTL credible-set leads


In [ ]:
def splice_junction_distance(phenotype_id, variant_pos):
    # phenotype_id: chr:start:end:clu_*:ENSG
    parts = str(phenotype_id).split(":")
    intron_start = int(parts[1])
    intron_end = int(parts[2])
    pos = int(variant_pos)
    return min(abs(pos - intron_start), abs(pos - intron_end))


parts = []
for tissue in tqdm(tissues, desc="sQTL SuSiE leads"):
    susie_path = sqtl_dir / f"{tissue}.v11.sQTLs.SuSiE_summary.parquet"
    pairs_path = sqtl_dir / f"{tissue}.v11.sQTLs.signif_pairs.parquet"
    if not susie_path.is_file():
        raise FileNotFoundError(f"Missing sQTL SuSiE summary: {susie_path}")
    if not pairs_path.is_file():
        raise FileNotFoundError(f"Missing sQTL signif pairs: {pairs_path}")

    susie = pd.read_parquet(
        susie_path,
        columns=["phenotype_id", "gene_id", "variant_id", "pip", "af", "cs_id", "cs_size"],
    )
    lead_idx = susie.groupby(["phenotype_id", "cs_id"], sort=False)["pip"].idxmax()
    leads = susie.loc[lead_idx].copy()
    leads["biotype"] = leads["gene_id"].map(id_to_biotype)
    leads = leads[leads["biotype"].isin(biotype_order)].copy()
    leads["variant_pos"] = leads["variant_id"].str.split("_").str[1].astype(int)

    pairs = pd.read_parquet(pairs_path, columns=["phenotype_id", "variant_id", "slope"])
    df = leads.merge(pairs, on=["phenotype_id", "variant_id"], how="left")
    df["dist_to_splice"] = [
        splice_junction_distance(pid, pos)
        for pid, pos in zip(df["phenotype_id"], df["variant_pos"])
    ]
    df["tissue"] = tissue
    parts.append(
        df[
            [
                "tissue",
                "phenotype_id",
                "gene_id",
                "variant_id",
                "cs_id",
                "cs_size",
                "pip",
                "af",
                "slope",
                "variant_pos",
                "dist_to_splice",
                "biotype",
            ]
        ]
    )

sqtl_df = pd.concat(parts, ignore_index=True)


### SuSiE sQTL credible-set leads


In [ ]:
print(f"sQTL CS-lead×tissue rows: {len(sqtl_df):,}")
print(sqtl_df["biotype"].value_counts().reindex(biotype_order).to_string())
print(
    f"missing slope: {sqtl_df['slope'].isna().sum():,} ({100 * sqtl_df['slope'].isna().mean():.1f}%)"
)


## Run analysis


In [ ]:
def add_plot_cols(df, kind):
    out = df.copy()
    out["maf"] = np.minimum(out["af"].astype(float), 1.0 - out["af"].astype(float))
    out["abs_slope"] = out["slope"].astype(float).abs()
    if kind == "eqtl":
        out["abs_dist"] = out["min_abs_tx_tss_dist"].astype(float)
        out["log10_abs_dist"] = np.log10(out["abs_dist"].clip(lower=1.0))
    else:
        out["abs_dist"] = out["dist_to_splice"].astype(float).abs()
        out["log10_abs_dist"] = np.log10(out["abs_dist"].clip(lower=1.0))
    out["biotype"] = pd.Categorical(out["biotype"], categories=biotype_order, ordered=True)
    return out


eqtl_plot = add_plot_cols(eqtl_df, "eqtl")
sqtl_plot = add_plot_cols(sqtl_df, "sqtl")

summary = (
    pd.concat(
        [
            eqtl_plot.assign(qtl="eqtl"),
            sqtl_plot.assign(qtl="sqtl"),
        ],
        ignore_index=True,
    )
    .groupby(["qtl", "biotype"], observed=True)
    .agg(
        n=("variant_id", "size"),
        maf_median=("maf", "median"),
        abs_slope_median=("abs_slope", "median"),
        abs_dist_median=("abs_dist", "median"),
    )
    .reset_index()
)
print(summary.to_string(index=False))


## Make figure


In [ ]:
def kde_by_biotype(ax, df, col, xlim=None, clip=None):
    for bt in biotype_order:
        sub = df.loc[df["biotype"] == bt, col].dropna().to_numpy()
        if len(sub) < kde_min_n:
            continue
        sns.kdeplot(
            sub,
            ax=ax,
            color=biotype_palette[bt],
            lw=1.1,
            label=biotype_pretty[bt],
            clip=clip,
            common_norm=False,
        )
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.set_ylabel("density")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


fig, axes = plt.subplots(2, 3, figsize=(fig_width, fig_height))
fig.subplots_adjust(left=0.10, right=0.98, bottom=0.10, top=0.95, wspace=0.45, hspace=0.55)

# eQTL: MAF, |β|, |TSS dist|
kde_by_biotype(axes[0, 0], eqtl_plot, "maf", xlim=(0, 0.5), clip=(0, 0.5))
axes[0, 0].set_xlabel("MAF")
axes[0, 0].set_ylabel("eQTL density")

kde_by_biotype(axes[0, 1], eqtl_plot, "abs_slope", xlim=(0, 2.5), clip=(0, None))
axes[0, 1].set_xlabel(r"$|\beta|$")

kde_by_biotype(axes[0, 2], eqtl_plot, "log10_abs_dist", xlim=(0, 6), clip=(0, None))
axes[0, 2].set_xlabel(r"$\log_{10}|\mathrm{TSS\ distance}|$ (bp)")

# sQTL: MAF, |β|, |splice dist|
kde_by_biotype(axes[1, 0], sqtl_plot, "maf", xlim=(0, 0.5), clip=(0, 0.5))
axes[1, 0].set_xlabel("MAF")
axes[1, 0].set_ylabel("sQTL density")

kde_by_biotype(axes[1, 1], sqtl_plot, "abs_slope", xlim=(0, 2.5), clip=(0, None))
axes[1, 1].set_xlabel(r"$|\beta|$")

kde_by_biotype(axes[1, 2], sqtl_plot, "log10_abs_dist", xlim=(0, 6), clip=(0, None))
axes[1, 2].set_xlabel(r"$\log_{10}|\mathrm{splice\ distance}|$ (bp)")

for ax in (axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]):
    ax.legend(frameon=False, loc="upper right", fontsize=4.5, handlelength=1.2, borderpad=0.2, labelspacing=0.2)
for ax in (axes[0, 2], axes[1, 2]):
    ax.legend(frameon=False, loc="upper left", fontsize=4.5, handlelength=1.2, borderpad=0.2, labelspacing=0.2)

fig.canvas.draw()
_inv = fig.transFigure.inverted()
for ax, label in [
    (axes[0, 0], "a"), (axes[0, 1], "b"), (axes[0, 2], "c"),
    (axes[1, 0], "d"), (axes[1, 1], "e"), (axes[1, 2], "f"),
]:
    _x = _inv.transform(ax.transAxes.transform((-0.15, 0)))[0]
    _y = _inv.transform(ax.transAxes.transform((0, 1.08)))[1]
    fig.text(_x, _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9)

plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "supp_figure_8_qtl_af_effect_size.png", dpi=fig_dpi, bbox_inches="tight")
fig.savefig(fig_dir / "supp_figure_8_qtl_af_effect_size.pdf", bbox_inches="tight")

## Print stats


In [ ]:
# No numeric values from this figure are cited in the paper text or caption.
